In [6]:

%matplotlib notebook
from importlib import reload

from scripts.mcf_reservoir_computing import mcf_reservoir_computing
reload(mcf_reservoir_computing)

from scripts.mcf_reservoir_computing.mcf_reservoir_computing import *


In [3]:
force_rerun = False  # игнорировать кэш и считать заново
save_cache = True
layer_count = 1
core_configuration = CoreConfig.hexagonal
core_count = get_core_count(core_configuration=core_configuration, ring_count=layer_count)

# layer 0 - центральная сердцевина
# layer 1 - первый круг из 6 сердцевин
# layer 2 - второй "круг" из 12 сердцевин, расстояние от которых до центра разное
# ...
layer_radii_array = np.zeros(int(layer_count) + 1)
for i in range(int(layer_count) + 1):
    if i == 0:
        layer_radii_array[i] = 0  # [mkm]
    if i == 1:
        layer_radii_array[i] = 30 # 17.3 * 1  # 17.3 # [mkm]
    if i == 2:
        layer_radii_array[i] = 17.3 * 2  # [mkm]
    if i == 3:
        layer_radii_array[i] = 17.3 * 3  # [mkm]

    # layer_radii_array[i] = 17.3 * (i * 1.5) # [mkm]

temporal_mask_modulation_frequency_ghz = 40  # GHz

variant = "temporal_same_all_cores"  # "spatial_only" "temporal_same_all_cores" "temporal_unique_per_core"

if variant == "temporal_same_all_cores" or variant == "temporal_unique_per_core":
    temporal_mask_size = 121
else:
    temporal_mask_size = 1

mg_cfg = MGConfig(t_size=2 ** 10, tau=17, n=10, beta=0.2, gamma=0.1, initial_condition=1.2, dt=1.0)

mask_cfg = MaskConfig(mask_size=temporal_mask_size, mask_kind="uniform", seed=42, gain_in=17.425874971099564)

reservoir_cfg = ReservoirConfig(
    fiber_length_m=0.6571013875307805,  # 0.1
    time_step_ps=1.0 / temporal_mask_modulation_frequency_ghz * 1e+3,
    step_number_per_dimensionless_distance=20,
    upsampling=2,
    delay_factor_in_symbols=6,
    delay_additional_in_mask_steps=0,
    layer_count=layer_count,
    layer_radii_array=layer_radii_array,
    g0_array=tuple([0.15163556871712178] * core_count),  # 10
    psat_array=tuple([1.0273745093098033e-05] * core_count),  # 0.02
    kappa=0.455,  # 0.9
    use_gpu=False,
    use_torch=False,
    num_threads=2,
    display_debug_plots=True,
    display_debug_info=True,
    save_figs=True,
    max_hours_total=72,
    precision='float64',
    use_dispersion=True,
    disable_core0=True,
)

training_cfg = TrainingConfig(feature_mode="intensity", taps=1, ridge_alpha=1e-4,  # washout=100,
                              target_shift=1,
                              train_frac=0.8, val_frac=0.2)  # для одиночного запуска

base_cfg = ExperimentConfig(core_count=core_count, mg=mg_cfg, mask=mask_cfg,
                            reservoir=reservoir_cfg, training=training_cfg,
                            variant=variant)

t_size = estimate_required_t_size_fast(base_cfg)
print("Estimated required symbol count =", t_size)
mg_cfg.t_size = 5000 # np.min([int(np.ceil(t_size / 500.0)) * 500, 5000])



Estimated required symbol count = 8771


In [4]:
# Пример: одиночный прогон с сохранением артефактов и pseudo free-run
if variant == "spatial_only":
    res = run_spatial_only(base_cfg, n_trials_opt=0, free_run_horizon=0, force_rerun=force_rerun, save_cache=save_cache)
elif variant == "temporal_same_all_cores":
    res = run_temporal_same_all_cores(base_cfg, n_trials_opt=0, free_run_horizon=0, force_rerun=force_rerun, save_cache=save_cache)
elif variant == "temporal_unique_per_core":
    res = run_temporal_unique_per_core(base_cfg, n_trials_opt=0, free_run_horizon=0, force_rerun=force_rerun, save_cache=save_cache)

print("Val/Test NRMSE:", res["metrics"]["nrmse_val"], res["metrics"]["nrmse_test"])



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

coupling_coefficient = 5.253341608151298
gamma = 0.00171823039805337
beta1 = 4892.8549256918905
beta2 = -7.75764312544366

Integral method = True
L_D        : 323.8 m
L_NL       : 1.579 m
L_coupling : 0.299 m
L_gain : 6.595 m

data_in.shape= (7, 605000)
data_in size = 15125000.0 ps
fiber_propagation_time = 3215.1017606589553 ps
feedback_loop_propagation_time=14934.9 ps
fiber_length_dimensionless = 2.197597486704675
length_scale = 0.29900898208439086
n_z = 44
esat =  [0.18646847 0.18646847 0.18646847 0.18646847 0.18646847 0.18646847
 0.18646847]

window_size * time_step_ps = 18150.0
eq.size = 7
  0   0   

0   0   0 

  0   0   



{'os_cpu_count': 16, 'env': {'OMP_NUM_THREADS': '2', 'MKL_NUM_THREADS': '2', 'OPENBLAS_NUM_THREADS': '2', 'VECLIB_MAXIMUM_THREADS': '2', 'OMP_WAIT_POLICY': 'ACTIVE', 'KMP_BLOCKTIME': 'infinite', 'KMP_AFFINITY': 'granularity=fine,compact,1,0', 'NUMBA_THREADING_LAYER': 'omp', 'NUMBA_NUM_THREADS': None}, 'numpy_blas_openmp': [{'user_api': 'blas', 'internal_api':

  5%|▍         | 2/44 [00:06<02:24,  3.43s/it]


KeyboardInterrupt: 

In [ ]:
# === Optuna: поиск по κ, g0, Psat, L_fiber и gain_in (лог по мощности) ===
# Вариант «temporal_same_all_cores» для одной сердцевины;
# окно задержки и размер маски тоже можно подстроить.
# Для запуска optuna dashboard установи
# pip install optuna-dashboard
# и выполни в отдельной консоли после запуска расчета (путь укажи свой до папки проекта)
# optuna-dashboard "C:/Users/Igor/YandexDisk/Code/Photonics/fiberprop/scripts/mcf_optuna.journal" --port 8080
# Открой в браузере http://127.0.0.1:8080/dashboard/
# На линуксе порт 18080, поэтому
# открой в браузере http://127.0.0.1:18080/dashboard

# base_cfg.training.train_frac = 0.8
# base_cfg.training.val_frac = 0.2
# res = run_temporal_same_all_cores(
#     base_cfg,
#     n_trials_opt=10000,  # сколько попробовать конфигураций
#     free_run_horizon=0,  # без свободного прогона после обучения
#     force_rerun=False,  # используй кэш, если ключ совпал
#     save_cache=True,
# )
# print("Лучший результат Optuna:\n", res)